### Physics Informed DeepONet for an integrator problem

As an introduction to *Operator Learning*, we consider the problem of learning an integrator operator using a Physics Informed DeepONet. This is a simple 1-dimensional problem where the goal is to learn the mapping $f \mapsto u$, where $u(x) = \int_0^x f(s) \, ds$.

In this setup, the input to the DeepONet is a function $f$ and a point $x$, and the output is the value of the integral $u(x)$. The network is trained to satisfy the integral relation by minimizing a physics-informed loss function.

In [ ]:
# This block is for GPU selection. Please execute.
# import pathlib
# import os
# user = int(str(pathlib.Path().resolve())[22:24])
# os.environ["CUDA_VISIBLE_DEVICES"]= str(user % 4)

path = "/localdata/tomfre/SolutionData/Integrator/"

In [ ]:
import qewton

# TODO: Define the variables for the Physics Informed DeepONet for the integrator problem
F = qewton.config.Variable
X = qewton.config.Variable
U = qewton.config.Variable

We load some data to have a class of functions available for training and testing the Physics Informed DeepONet for the integrator problem.

The **u_data** is only used for testing the accuracy of the learned operator, but it is *not required* during the training of the Physics Informed DeepONet.

In [ ]:
# Loading the data
f_data = qewton.DEFAULT_DL_BACKEND.load_data(path + "f_data.pt")
u_data = qewton.DEFAULT_DL_BACKEND.load_data(path + "u_data.pt")
x_data = qewton.DEFAULT_DL_BACKEND.load_data(path + "x_data.pt")

# Splitting the data into training and testing sets
train_f_data, _, train_x_data = f_data[:4000], u_data[:4000], x_data[:4000]
test_f_data, test_u_data, test_x_data = f_data[4000:], u_data[4000:], x_data[4000:]

batch_size = 1000

In [ ]:
# Next we build the datasets and data loaders for training and testing

dataset = qewton.data.GridDataSet(
    data=[train_f_data, train_x_data], 
    feature_variables=[F, X],
    geometry_variable=X,
    point_grid=x_data[0]
)

test_dataset = qewton.data.GridDataSet(
    data=[test_f_data, test_x_data, test_u_data],
    feature_variables=[F, X, U],
    geometry_variable=X,
    point_grid=x_data[0]
)

data_loader = qewton.data.DataLoader(
    data_set=dataset,
    test_data_set=test_dataset,
    batch_size=batch_size,
    splitting_ratio=(1.0, 0.0, 0.0),
    shuffle_data=False,
)

Now let us define the DeepONet for the integrator problem.
As explained the architecture of the DeepONet consists of a branch network that takes the function values as input and a trunk network that takes the spatial coordinates as input. The outputs of these networks are then combined to produce the final prediction.

We will now define the branch and trunk networks using fully connected layers.

In [ ]:
# TODO: Add the correct input variables for trunk and branch
# TODO: set both the trunk and branch hidden neurons to 16
model = qewton.algorithms.FCNDeepONet(
    trunk_input=, 
    branch_input=,
    output=U,
    trunk_hidden_neurons=,
    branch_hidden_neurons=,
    trunk_hidden_layers=3,
    branch_hidden_layers=2,
    intermediate_neurons=32,
    activations=qewton.bb.Tanh
)

Now the residuals can be defined as before.

In [ ]:
# TODO: Build the residual for the ODE
def ode_residual(x: X, f: F, u: U):
    return

def initial_residual(u: U):
    return u[:, 0]


For the computation pipeline we now combine both residuals into one pipeline.

In [ ]:
# First build a MSE constraint for the ODE and initial conditions
ode_constraint = qewton.constraints.MSEConstraint()
initial_constraint = qewton.constraints.MSEConstraint(weight=50.0, name="initial_constraint")

pipeline = qewton.Graph()

with pipeline.tracker(0):
    f, x, u = data_loader()
    x = x.track_gradient()
    model_out = # TODO: evaluate the model at (f, x)
    ode_res = # TODO: compute the ODE residual using §ode_residual"
    initial_res = # TODO: compute the initial condition residual using "initial_residual"
    ode_constraint(ode_res, 0.0)
    initial_constraint(initial_res, 0.0)

Now a short training to see everything works:

In [ ]:
adam_phase = qewton.optim.OptimizationPhase(
    optimizer=qewton.optim.Adam(),
    lr=0.01,
    max_iterations=2000,
    lr_scheduler=qewton.optim.StepLR(step_size=2000, gamma=0.25)
)

trainer = qewton.optim.GraphBasedTrainer(
    optimization_phases=[adam_phase],
    graphs=[pipeline],
    training_objectives=[ode_constraint],
    device=qewton.cuda(0),
)

trainer.run()

Plot the solution:

In [ ]:
layout = pipeline.visualize(
    model.output_ports[0],
    reference=data_loader.get_output_port(U),
    mode=qewton.optim.EvaluationPhase.TEST,
)
fig = qewton.viz.Figure(layout)
fig.show()